In [4]:
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn as nn
import torch.optim as optim
import torch_geometric
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool
from torch_cluster import radius_graph
from torch_scatter import scatter_add
from sklearn.model_selection import train_test_split
from ase import Atoms
from ase.data import atomic_numbers
from e3nn import o3
from e3nn.o3 import Irreps, FullyConnectedTensorProduct
from e3nn.nn import Gate
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from mpl_toolkits.mplot3d import Axes3D

# =============================================================================
# CONFIGURATION
# =============================================================================
DEBUG = True
CUTOFF_RADIUS = 5.0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PLOT_DIR = "../training_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

# =============================================================================
# DEBUG UTILITIES
# =============================================================================
def debug_print(title, data=None, tensors=None, stats=False):
    if DEBUG:
        separator = "\n" + "="*80
        print(separator)
        print(f"DEBUG: {title}")
        
        if data is not None:
            print("- Data Details:")
            if isinstance(data, dict):
                for k, v in data.items():
                    print(f"  {k}: {type(v)} | Content: {v}")
            else:
                print(f"- Type: {type(data)}")
                print(f"- Content: {data}")
        
        if tensors is not None:
            print("- Tensor Details:")
            if isinstance(tensors, dict):
                for name, tensor in tensors.items():
                    _print_tensor_info(name, tensor, stats)
            else:
                _print_tensor_info("Tensor", tensors, stats)
        
        print(separator + "\n")

def _print_tensor_info(name, tensor, stats=False):
    if tensor is None:
        print(f"  {name}: None")
        return
    
    print(f"  {name}:")
    print(f"    Shape: {tensor.shape}")
    print(f"    Dtype: {tensor.dtype}")
    print(f"    Device: {tensor.device}")
    
    if stats and tensor.numel() > 0:
        t = tensor.detach().cpu().float()
        print(f"    Min: {t.min().item():.4f}")
        print(f"    Max: {t.max().item():.4f}")
        print(f"    Mean: {t.mean().item():.4f}")
        print(f"    Std: {t.std().item():.4f}")
        print(f"    NaN: {torch.isnan(t).any().item()}")
        print(f"    Inf: {torch.isinf(t).any().item()}")
    else:
        if tensor.numel() > 10:
            print(f"    Values (first 10): {tensor.flatten()[:10]}")
        else:
            print(f"    Values: {tensor}")

# =============================================================================
# VISUALIZATION UTILITIES
# =============================================================================
def plot_loss_curves(train_losses, val_losses, epoch):
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title(f'Loss Curves - Epoch {epoch}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{PLOT_DIR}/loss_epoch_{epoch}.png")
    plt.close()

def plot_predictions_vs_actuals(y_true, y_pred, epoch):
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x=y_true, y=y_pred)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    plt.title(f'Predictions vs Actuals - Epoch {epoch}')
    plt.xlabel('Actual Values')
    plt.ylabel('Predictions')
    plt.savefig(f"{PLOT_DIR}/predictions_epoch_{epoch}.png")
    plt.close()

def plot_molecule_3d(pos, symbols, edge_index, mol_id, inchi=None):
    """3D visualization of molecular structure with edges and InChI Key."""
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    pos = pos.cpu().numpy() if isinstance(pos, torch.Tensor) else pos
    edge_index = edge_index.cpu().numpy() if isinstance(edge_index, torch.Tensor) else edge_index
    
    # Plot atoms
    ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c='blue', s=100, 
               depthshade=True, alpha=0.8)
    
    # Annotate atoms
    for i, (x, y, z) in enumerate(pos):
        ax.text(x, y, z, symbols[i], 
                ha='center', va='center', 
                fontsize=9, color='white', 
                weight='bold', bbox=dict(facecolor='black', alpha=0.5))
    
    # Plot edges
    for src, dst in edge_index.T:
        ax.plot(*zip(pos[src], pos[dst]), color='gray', 
                linewidth=1, alpha=0.4)
    
    # (MODIFICADO) Actualizamos el título para usar InChI Key
    title = f'3D Molecular Structure - Molecule {mol_id}'
    if inchi and inchi != 'Unknown InChI':
        title += f'\nInChI Key: {inchi}'
    else:
        title += '\nIdentifier: Molecule Index'
    
    ax.set_title(title)
    ax.set_xlabel('X (Å)')
    ax.set_ylabel('Y (Å)')
    ax.set_zlabel('Z (Å)')
    
    # (MODIFICADO) Generamos un nombre de archivo único usando la InChI Key
    filename = f"mol_{inchi}" if inchi and inchi != 'Unknown InChI' else f"mol_{mol_id}"
    # Reemplazamos caracteres problemáticos como '/'
    filename = filename.replace("/", "_")

    # Guardamos la figura
    plt.savefig(f"{PLOT_DIR}/{filename}.png")
    plt.close()

# =============================================================================
# DATA PROCESSING
# =============================================================================
def _safe_float(val):
    """Safe conversion to float"""
    try:
        return float(val) if val is not None else None
    except:
        return None

def _safe_list_float(val):
    """Safe conversion of list elements to floats"""
    try:
        return [float(x) for x in val] if isinstance(val, list) else None
    except:
        return None

def _extract_irreps(irreps: o3.Irreps, l=None, p=None):
    """Filter irreps by angular momentum and parity"""
    return o3.Irreps([
        (mul, (ell, parity))
        for mul, (ell, parity) in irreps
        if (l is None or ell == l) and (p is None or parity == p)
    ])

# =============================================================================
# DATASET CLASS
# =============================================================================
class MolecularDataset(Dataset):
    def __init__(self, data_list, scaler=None):
        super().__init__()
        self.data_list = data_list
        self.scaler = scaler
        self._precompute_normalization()
        self._validate_data()

    def _validate_data(self):
        """Filter invalid molecules"""
        valid_indices = []
        for idx in tqdm(range(len(self.data_list)), desc="Validating data"):
            try:
                if self.get(idx) is not None:
                    valid_indices.append(idx)
            except Exception as e:
                debug_print(f"Error in molecule {idx}", data=str(e))
        self.data_list = [self.data_list[i] for i in valid_indices]
        
        if not self.data_list:
            raise ValueError("No valid molecules found in dataset!")
        
        debug_print("Data Validation", data={
            'original_samples': len(self.data_list),
            'valid_samples': len(valid_indices)
        })

    def _precompute_normalization(self):
        """Initialize feature scalers"""
        if self.scaler:
            self.homo_scaler = self.scaler['homo']
            self.lumo_scaler = self.scaler['lumo']
            self.dipole_scaler = self.scaler['dipole']
            return

        # Collect features for scaling
        features = {
            'homo': [],
            'lumo': [],
            'dipole': []
        }
        
        for mol in self.data_list:
            features['homo'].append(_safe_float(mol.get('homo (eV)')) or 0.0)
            features['lumo'].append(_safe_float(mol.get('lumo (eV)')) or 0.0)
            dipole = _safe_list_float(mol.get('dipole_moment_vector_S1 (D)')) or [0.0]*3
            features['dipole'].append(dipole)
        
        # Create scalers
        self.homo_scaler = StandardScaler().fit(np.array(features['homo']).reshape(-1, 1))
        self.lumo_scaler = StandardScaler().fit(np.array(features['lumo']).reshape(-1, 1))
        self.dipole_scaler = StandardScaler().fit(np.array(features['dipole']))
        
        debug_print("Feature Scalers", data={
            'HOMO': f"μ: {self.homo_scaler.mean_[0]:.2f} σ: {self.homo_scaler.scale_[0]:.2f}",
            'LUMO': f"μ: {self.lumo_scaler.mean_[0]:.2f} σ: {self.lumo_scaler.scale_[0]:.2f}",
            'Dipole': f"μ: {self.dipole_scaler.mean_} σ: {self.dipole_scaler.scale_}"
        })

    def len(self):
        return len(self.data_list)

    def get(self, idx):
        """Convert raw data to PyG Data object"""
        try:
            mol = self.data_list[idx]
            
            # (MODIFICADO) Usamos inchi_key en lugar de 'inchi'
            inchi_key = mol.get('inchi_key', f'Unknown_InChI_Key_{idx}')

            # Target value
            target = _safe_float(mol.get('reduction_potential_S1 (eV)'))
            if target is None:
                return None

            # Parse molecular structure
            coord_data = mol.get("optimized_coordinates_S0", [])
            if len(coord_data) < 2:
                return None
            
            n_atoms = int(coord_data[0])
            coord_lines = coord_data[1:n_atoms+1]
            
            symbols, positions = [], []
            for line in coord_lines:
                parts = line.strip().split()
                if len(parts) >= 4:
                    symbols.append(parts[0])
                    positions.append(list(map(float, parts[1:4])))
            
            if not symbols:
                return None

            # Create ASE Atoms object
            atoms = Atoms(symbols=symbols, positions=positions)
            pos = torch.tensor(atoms.positions, dtype=torch.float32).to(DEVICE)
            
            # Generate graph edges
            edge_index = radius_graph(pos, r=CUTOFF_RADIUS)

            # (MODIFICADO) Graficamos SIEMPRE cada molécula con su inchi_key
            plot_molecule_3d(pos, symbols, edge_index, idx, inchi_key)

            # Node features (atomic numbers)
            node_z = torch.tensor([atomic_numbers[s] for s in symbols], 
                                  dtype=torch.long).unsqueeze(-1).to(DEVICE)

            # Edge attributes
            row, col = edge_index
            edge_vec = pos[col] - pos[row]
            dist = torch.norm(edge_vec, dim=1, keepdim=True)
            rbf = torch.exp(-(dist / CUTOFF_RADIUS)**2)
            edge_attr = torch.cat([rbf, edge_vec/dist.clamp(min=1e-9)], dim=-1).to(DEVICE)

            # Electronic properties
            homo = self.homo_scaler.transform([[mol.get('homo (eV)') or 0.0]]).flatten()
            lumo = self.lumo_scaler.transform([[mol.get('lumo (eV)') or 0.0]]).flatten()
            dipole = self.dipole_scaler.transform(
                [mol.get('dipole_moment_vector_S1 (D)') or [0.0]*3]
            ).flatten()

            # Create Data object
            data = Data(
                x=node_z,
                pos=pos,
                edge_index=edge_index,
                edge_attr=edge_attr,
                y=torch.tensor([target], dtype=torch.float32).to(DEVICE),
                dipole=torch.tensor(dipole, dtype=torch.float32).to(DEVICE),
                homo=torch.tensor(homo, dtype=torch.float32).to(DEVICE),
                lumo=torch.tensor(lumo, dtype=torch.float32).to(DEVICE),
                batch=torch.zeros(node_z.size(0), dtype=torch.long).to(DEVICE)
            )

            debug_print(f"Molecule {idx}", tensors={
                'positions': pos,
                'edge_index': edge_index,
                'edge_attributes': edge_attr
            })
            
            return data
        
        except Exception as e:
            debug_print(f"Error processing molecule {idx}", data=str(e))
            return None

# =============================================================================
# NEURAL NETWORK COMPONENTS
# =============================================================================
class InteractionBlock(nn.Module):
    def __init__(self, node_irreps="2x0e+1x1o", edge_irreps="1x0e+1x1o"):
        super().__init__()
        self.node_irreps = o3.Irreps(node_irreps)
        self.edge_irreps = o3.Irreps(edge_irreps)
        
        self.tensor_product = FullyConnectedTensorProduct(
            irreps_in1=self.node_irreps,
            irreps_in2=self.edge_irreps,
            irreps_out=self.node_irreps,
            internal_weights=True,
            shared_weights=True
        )
        
        # Gate components
        scalars = _extract_irreps(self.node_irreps, l=0, p=1)
        vectors = _extract_irreps(self.node_irreps, l=1, p=-1)
        gate_irreps = "1x0e" if vectors.dim > 0 else "0e"
        
        self.gate = Gate(
            irreps_scalars=scalars,
            act_scalars=[nn.SiLU() for _ in range(len(scalars))],
            irreps_gates=gate_irreps,
            act_gates=[nn.Sigmoid()],
            irreps_gated=vectors
        )
        
        self.norm = nn.LayerNorm(self.node_irreps.dim)
        
        debug_print("Interaction Block", data={
            'input_irreps': node_irreps,
            'output_dim': self.node_irreps.dim
        })

    def forward(self, x, edge_attr, edge_index):
        src, dst = edge_index
        messages = self.tensor_product(x[src], edge_attr)
        messages = self.gate(messages)
        aggregated = scatter_add(messages, dst, dim=0, dim_size=x.size(0))
        return self.norm(x + aggregated)

class MolecularGNN(nn.Module):
    def __init__(self, num_layers=3):
        super().__init__()
        self.embedding = nn.Embedding(100, 5)  # Embed atomic numbers
        self.layers = nn.ModuleList([
            InteractionBlock("2x0e+1x1o", "1x0e+1x1o") 
            for _ in range(num_layers)
        ])
        
        debug_print("GNN Architecture", data={
            'num_layers': num_layers,
            'embedding_dim': 5
        })

    def forward(self, x, edge_index, edge_attr):
        x = self.embedding(x.squeeze(-1))
        for layer in self.layers:
            x = layer(x, edge_attr, edge_index)
        return x

class ReductionPotentialPredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.gnn = MolecularGNN(num_layers=3)

        # Electronic properties processor
        self.electronic_net = nn.Sequential(
            nn.Linear(5, 32),  # dipole(3) + homo(1) + lumo(1)
            nn.SiLU(),
            nn.Linear(32, 16),
            nn.Dropout(0.1)
        )

        # Combined predictor
        self.predictor = nn.Sequential(
            nn.Linear(16 + 5, 64),  # GNN features (5) + electronic (16)
            nn.SiLU(),
            nn.Linear(64, 32),
            nn.SiLU(),
            nn.Linear(32, 1)
        )

        debug_print("Predictor Architecture", data={
            'total_params': sum(p.numel() for p in self.parameters())
        })

    def forward(self, data):
        # Structural features
        structural = self.gnn(data.x, data.edge_index, data.edge_attr)
        structural_pool = global_mean_pool(structural, data.batch)

        # Electronic features
        electronic = torch.cat([data.dipole, data.homo, data.lumo], dim=1)
        electronic = self.electronic_net(electronic)

        # Combined prediction
        combined = torch.cat([structural_pool, electronic], dim=1)
        return self.predictor(combined)

# =============================================================================
# TRAINING FRAMEWORK
# =============================================================================
class ModelTrainer:
    def __init__(self, config):
        self.config = config
        self.train_losses = []
        self.val_losses = []
        self._init_data()
        self._init_model()

    def _init_data(self):
        with open(self.config['data_path']) as f:
            full_data = json.load(f)

        train_data, val_data = train_test_split(
            full_data,
            test_size=self.config['test_size'],
            random_state=self.config['seed']
        )

        self.train_set = MolecularDataset(train_data)
        self.val_set = MolecularDataset(val_data, scaler={
            'homo': self.train_set.homo_scaler,
            'lumo': self.train_set.lumo_scaler,
            'dipole': self.train_set.dipole_scaler
        })

        self.train_loader = DataLoader(
            self.train_set,
            batch_size=self.config['batch_size'],
            shuffle=True,
            collate_fn=lambda batch: torch_geometric.data.Batch.from_data_list(
                [b for b in batch if b]
            )
        )

        self.val_loader = DataLoader(
            self.val_set,
            batch_size=self.config['batch_size'],
            collate_fn=lambda batch: torch_geometric.data.Batch.from_data_list(
                [b for b in batch if b]
            )
        )

        debug_print("Data Loaders", data={
            'train_samples': len(self.train_set),
            'validation_samples': len(self.val_set),
            'batch_size': self.config['batch_size']
        })

    def _init_model(self):
        self.model = ReductionPotentialPredictor().to(DEVICE)
        self.optimizer = optim.AdamW(
            self.model.parameters(),
            lr=self.config['lr'],
            weight_decay=self.config['weight_decay']
        )
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode='min',
            factor=0.5,
            patience=3,
            verbose=True
        )
        self.loss_fn = nn.HuberLoss()

        debug_print("Training Setup", data={
            'model': self.model.__class__.__name__,
            'optimizer': self.optimizer.__class__.__name__,
            'loss_function': self.loss_fn.__class__.__name__
        })

    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0.0
        progress = tqdm(self.train_loader, desc=f"Training Epoch {epoch}")

        for batch in progress:
            batch = batch.to(DEVICE)
            self.optimizer.zero_grad()

            pred = self.model(batch)
            loss = self.loss_fn(pred, batch.y)

            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()

            total_loss += loss.item() * batch.num_graphs
            progress.set_postfix({'loss': loss.item()})

            if DEBUG and progress.n % 10 == 0:
                debug_print("Training Batch", data={
                    'current_loss': loss.item(),
                    'learning_rate': self.optimizer.param_groups[0]['lr']
                })

        return total_loss / len(self.train_set)

    def evaluate(self, loader, epoch):
        self.model.eval()
        total_loss = 0.0
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for batch in tqdm(loader, desc=f"Validation Epoch {epoch}"):
                batch = batch.to(DEVICE)
                pred = self.model(batch)
                loss = self.loss_fn(pred, batch.y)

                total_loss += loss.item() * batch.num_graphs
                all_preds.append(pred.cpu())
                all_targets.append(batch.y.cpu())

        all_preds = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()

        plot_predictions_vs_actuals(all_targets, all_preds, epoch)
        return total_loss / len(loader.dataset), all_preds, all_targets

    def run_training(self):
        best_loss = float('inf')

        for epoch in range(self.config['epochs']):
            train_loss = self.train_epoch(epoch)
            val_loss, preds, targets = self.evaluate(self.val_loader, epoch)

            self.scheduler.step(val_loss)
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)

            plot_loss_curves(self.train_losses, self.val_losses, epoch)

            if val_loss < best_loss:
                best_loss = val_loss
                torch.save(self.model.state_dict(), self.config['save_path'])
                debug_print("Model Checkpoint", data={
                    'validation_loss': val_loss,
                    'checkpoint_path': self.config['save_path']
                })

            debug_print("Epoch Summary", data={
                'epoch': epoch,
                'train_loss': train_loss,
                'validation_loss': val_loss,
                'learning_rate': self.optimizer.param_groups[0]['lr']
            })

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    config = {
        'data_path': 'fd10.json',
        'seed': 42,
        'test_size': 0.2,
        'batch_size': 16,
        'lr': 1e-3,
        'weight_decay': 1e-5,
        'epochs': 50,
        'save_path': 'best_model.pth'
    }

    trainer = ModelTrainer(config)
    trainer.run_training()


FileNotFoundError: [Errno 2] No such file or directory: 'fd10.json'